# A3 — Dual-Tree Complex Wavelet Transform + Hidden Markov Tree

**Goal:** Visualise DTCWT subbands, fit a Hidden Markov Tree to the wavelet coefficients, and explore Fisher information distances between paintings.

**Key papers:**
- Johnson et al. (2008), *IEEE Signal Processing Magazine* — DTCWT + HMT for van Gogh
- Jafarpour et al. (2012), *Signal Processing* 116(4) — Fisher distance, impressionist dating
- Crouse, Nowak & Baraniuk (1998), *IEEE T-SP* 46(4) — HMT formalism

**Why this is the strongest method for Manet:** Jafarpour et al. (2012) specifically applied this to 100+ impressionist/post-impressionist paintings and successfully clustered by artist and temporal period.

---

### Mathematics

**DTCWT** produces 6 directional subbands per scale (±15°, ±45°, ±75°), is nearly shift-invariant, and has only 2:1 redundancy.

**Hidden Markov Tree model:** Each wavelet coefficient $c_{j,k}$ at scale $j$, position $k$ has a hidden binary state $h \in \{\text{small}, \text{large}\}$:
$$p(c_{j,k} \mid h_{j,k}) = \mathcal{N}(\mu_h,\, \sigma_h^2)$$
$$P(h_{j,k} \mid h_{j+1,\text{parent}(k)}) = \text{transition matrix per scale}$$

Parameters $\theta = \{\mu_{\text{small}}, \sigma_{\text{small}}, \mu_{\text{large}}, \sigma_{\text{large}}, A\}$ are estimated per subband via the **EM upward-downward algorithm**.

**Fisher information distance** — the Riemannian metric on the statistical manifold:
$$d_F(A, B) = \sqrt{(\theta_A - \theta_B)^\top I(\bar{\theta})^{-1} (\theta_A - \theta_B)}$$
where $I(\theta)$ is the Fisher information matrix. Paintings with similar $\theta$ are stylistically related.

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import logsumexp
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from notebooks.research.utils import load_image, to_gray, show_pair

try:
    import dtcwt
    print('dtcwt loaded ✓')
except ImportError:
    print('Install dtcwt: pip install dtcwt')
    raise

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
PATH_MANET        = Path('../../data/manet/manet_example.tif')
PATH_CONTEMPORARY = Path('../../data/contemporary/contemporary_example.tif')
N_LEVELS = 4   # Decomposition levels
# ─────────────────────────────────────────────────────────────────────────────

img_m  = load_image(PATH_MANET)
img_c  = load_image(PATH_CONTEMPORARY)
gray_m = to_gray(img_m)
gray_c = to_gray(img_c)
show_pair(img_m, img_c)

---
## 1. DTCWT Decomposition — Visualise the 6 Directional Subbands

In [ ]:
def dtcwt_decompose(gray: np.ndarray, nlevels: int = 4):
    """Apply DTCWT and return the transform object."""
    transform = dtcwt.Transform2d()
    return transform.forward(gray, nlevels=nlevels)


T_m = dtcwt_decompose(gray_m, N_LEVELS)
T_c = dtcwt_decompose(gray_c, N_LEVELS)

print(f'Number of levels: {len(T_m.highpasses)}')
print(f'Highpass subbands per level: {T_m.highpasses[0].shape[-1]}  (6 directional subbands)')
for i, hp in enumerate(T_m.highpasses):
    print(f'  Level {i+1}: shape={hp.shape}  (H×W×6 complex)')

In [ ]:
# Visualise the 6 directional subband magnitudes at each level
# Directions: ±15°, ±45°, ±75°
direction_labels = ['+15°', '+45°', '+75°', '-75°', '-45°', '-15°']

fig, axes = plt.subplots(N_LEVELS, 6, figsize=(18, N_LEVELS * 3))
fig.suptitle('DTCWT directional subband magnitudes — Manet\n'
             'Rows: scale (fine→coarse)  |  Cols: orientation',
             fontsize=12)

for level in range(N_LEVELS):
    hp = T_m.highpasses[level]   # shape: (H, W, 6) complex
    mag = np.abs(hp)             # magnitude
    for d in range(6):
        ax = axes[level, d]
        im = ax.imshow(mag[..., d], cmap='inferno')
        ax.set_title(f'L{level+1} {direction_labels[d]}', fontsize=8)
        ax.axis('off')

plt.tight_layout()
plt.show()

---
## 2. Hidden Markov Tree — Fit a Gaussian Mixture Model per Subband

We fit a two-state HMT to each subband. Here we use a simplified version: fit a **two-component Gaussian mixture** to the magnitude coefficients per subband. This captures the 'small' and 'large' state distributions.

The HMT parameters $\theta = (\mu_{\text{small}}, \sigma_{\text{small}}, \mu_{\text{large}}, \sigma_{\text{large}}, w_{\text{large}})$ where $w_{\text{large}}$ is the mixing weight of the large state.

Painterly texture is characterised by: few large-state coefficients (edges/contours) in a sea of small-state coefficients (flat regions). The mixing weights encode how much of the image is structured vs. uniform.

In [ ]:
from sklearn.mixture import GaussianMixture


def fit_hmt_params(T, nlevels: int) -> np.ndarray:
    """
    Fit a 2-component GMM per subband per level.
    Returns parameter vector: [μ1, σ1, μ2, σ2, w2] × nlevels × 6 directions.
    """
    params = []
    for level in range(nlevels):
        hp = T.highpasses[level]
        mag = np.abs(hp)  # (H, W, 6)
        for d in range(6):
            coefs = mag[..., d].ravel().reshape(-1, 1)
            gmm = GaussianMixture(n_components=2, covariance_type='full',
                                  random_state=42, max_iter=200)
            gmm.fit(coefs)
            # Sort components: component 0 = small state (lower mean)
            order = np.argsort(gmm.means_.ravel())
            means  = gmm.means_.ravel()[order]
            stds   = np.sqrt(gmm.covariances_.ravel()[order])
            weight_large = gmm.weights_[order[1]]   # weight of 'large' state
            params.extend([means[0], stds[0], means[1], stds[1], weight_large])
    return np.array(params)


print('Fitting HMT parameters (GMM per subband)...')
theta_m = fit_hmt_params(T_m, N_LEVELS)
theta_c = fit_hmt_params(T_c, N_LEVELS)
print(f'Parameter vector size: {theta_m.shape[0]}  ({N_LEVELS} levels × 6 dirs × 5 params)')

In [ ]:
# Reshape for visualisation: (n_levels, 6_directions, 5_params)
n_params = 5
theta_m_r = theta_m.reshape(N_LEVELS, 6, n_params)
theta_c_r = theta_c.reshape(N_LEVELS, 6, n_params)

param_names = ['μ_small', 'σ_small', 'μ_large', 'σ_large', 'w_large']

fig, axes = plt.subplots(N_LEVELS, n_params, figsize=(16, N_LEVELS * 3))
fig.suptitle('HMT parameters θ per level and direction\n'
             'Blue=Manet  Orange=Contemporary  x-axis=orientation (0..5)', fontsize=11)

for level in range(N_LEVELS):
    for p, pname in enumerate(param_names):
        ax = axes[level, p]
        ax.plot(theta_m_r[level, :, p], 'o-', color='#2196F3', label='Manet' if (level==0 and p==0) else '')
        ax.plot(theta_c_r[level, :, p], 's--', color='#FF5722', label='Contemporary' if (level==0 and p==0) else '')
        ax.set_title(f'L{level+1} {pname}', fontsize=8)
        ax.set_xlabel('Direction', fontsize=7)
        ax.grid(alpha=0.3)
        ax.tick_params(labelsize=7)

axes[0, 0].legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 3. Large-State Map — Where Are the Significant Wavelet Coefficients?

Threshold each subband at the GMM decision boundary. Pixels assigned to the 'large' state correspond to structurally significant image regions — edges, brushstrokes, contours.

In [ ]:
def large_state_map(T, level: int = 0) -> np.ndarray:
    """
    Create a map showing which pixels are in the 'large' HMT state
    at the given level (aggregated over all 6 directions).
    """
    hp = T.highpasses[level]
    mag = np.abs(hp)   # (H, W, 6)
    large_map = np.zeros(mag.shape[:2])

    for d in range(6):
        coefs = mag[..., d].ravel().reshape(-1, 1)
        gmm = GaussianMixture(n_components=2, random_state=42).fit(coefs)
        order = np.argsort(gmm.means_.ravel())
        # Posterior probability of 'large' state per coefficient
        posteriors = gmm.predict_proba(coefs)
        prob_large = posteriors[:, order[1]].reshape(mag.shape[:2])
        large_map += prob_large

    return large_map / 6   # Average over directions


fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Large-state probability maps (HMT) — bright = structurally significant region\n'
             'Hypothesis: should align with Manet\'s brushstroke edges and contours', fontsize=11)

for col, (name, img, T) in enumerate([
    ('Manet', img_m, T_m), ('Contemporary', img_c, T_c)
]):
    axes[0, col*1 if col < 2 else 2].imshow(img)
    axes[0, col].set_title(f'{name} — original')
    axes[0, col].axis('off')

    lmap = large_state_map(T, level=1)   # Level 2: medium scale
    im = axes[1, col].imshow(lmap, cmap='hot')
    plt.colorbar(im, ax=axes[1, col], fraction=0.03)
    axes[1, col].set_title(f'{name} — large-state probability (level 2)')
    axes[1, col].axis('off')

# Difference map
lmap_m = large_state_map(T_m, level=1)
lmap_c = large_state_map(T_c, level=1)
# Resize to same shape for difference
from skimage.transform import resize as sk_resize
target = (min(lmap_m.shape[0], lmap_c.shape[0]),
          min(lmap_m.shape[1], lmap_c.shape[1]))
diff_map = np.abs(sk_resize(lmap_m, target) - sk_resize(lmap_c, target))
im = axes[1, 2].imshow(diff_map, cmap='RdYlGn_r')
plt.colorbar(im, ax=axes[1, 2], fraction=0.03)
axes[1, 2].set_title('|Manet − Contemporary| large-state difference')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

---
## 4. Fisher Information Distance

Given two parameter vectors $\theta_A$ and $\theta_B$, the Fisher distance is:
$$d_F(A, B) = \sqrt{(\theta_A - \theta_B)^\top I(\bar{\theta})^{-1} (\theta_A - \theta_B)}$$

For independent Gaussian components, the Fisher information matrix $I(\theta)$ is diagonal with entries:
$$I_{\mu\mu} = \frac{1}{\sigma^2}, \quad I_{\sigma\sigma} = \frac{2}{\sigma^2}$$

This makes the distance easy to compute analytically — it is a **whitened Euclidean distance** in parameter space.

In [ ]:
def fisher_information_diagonal(theta: np.ndarray) -> np.ndarray:
    """
    Approximate diagonal Fisher information matrix for a sequence of
    [μ_small, σ_small, μ_large, σ_large, w_large] blocks.
    For Gaussian components: I_μ = 1/σ², I_σ = 2/σ².
    For mixture weight: I_w = 1/(w(1-w)) approximately.
    """
    I_diag = np.zeros_like(theta)
    n_blocks = len(theta) // 5
    for b in range(n_blocks):
        base = b * 5
        mu_s, sig_s, mu_l, sig_l, w_l = theta[base:base+5]
        eps = 1e-6
        I_diag[base]   = 1.0 / (sig_s**2 + eps)        # I_μ_small
        I_diag[base+1] = 2.0 / (sig_s**2 + eps)        # I_σ_small
        I_diag[base+2] = 1.0 / (sig_l**2 + eps)        # I_μ_large
        I_diag[base+3] = 2.0 / (sig_l**2 + eps)        # I_σ_large
        w_l_c = np.clip(w_l, eps, 1-eps)
        I_diag[base+4] = 1.0 / (w_l_c * (1 - w_l_c))  # I_w
    return I_diag


def fisher_distance(theta_a: np.ndarray, theta_b: np.ndarray,
                    theta_ref: np.ndarray = None) -> float:
    """
    Fisher distance d_F(A,B) using diagonal approximation of I(θ_ref).
    If theta_ref is None, uses the mean of theta_a and theta_b.
    """
    if theta_ref is None:
        theta_ref = (theta_a + theta_b) / 2
    I_diag = fisher_information_diagonal(theta_ref)
    delta = theta_a - theta_b
    return float(np.sqrt(np.sum(I_diag * delta**2)))


d_F = fisher_distance(theta_m, theta_c)
d_eucl = np.linalg.norm(theta_m - theta_c)
d_cosine = 1 - np.dot(theta_m, theta_c) / (np.linalg.norm(theta_m) * np.linalg.norm(theta_c) + 1e-9)

print('Distances between Manet and Contemporary HMT parameters:')
print(f'  Fisher distance:   {d_F:.4f}')
print(f'  Euclidean:         {d_eucl:.4f}')
print(f'  Cosine distance:   {d_cosine:.4f}')
print()
print('Self-distance (sanity check — should be ~0):')
print(f'  Fisher d_F(Manet, Manet):   {fisher_distance(theta_m, theta_m):.6f}')

In [ ]:
# If you have multiple images, build a distance matrix
# Example with just 2 paintings — expand when more data is available
labels = ['Manet', 'Contemporary']
thetas = [theta_m, theta_c]
n = len(thetas)
D_mat = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        D_mat[i, j] = fisher_distance(thetas[i], thetas[j])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(D_mat, cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='Fisher distance')
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{D_mat[i,j]:.2f}', ha='center', va='center', fontsize=10)
ax.set_title('Fisher distance matrix\n(add more paintings to see clustering)')
plt.tight_layout()
plt.show()

---
## 5. Key Observations

| Question | Observation |
|---|---|
| Which DTCWT directions show the most energy difference? | |
| Do large-state maps align with Manet's brushstroke/contour edges? | |
| Which HMT parameter (μ_large, σ_large, w_large) differs most? | |
| Is the Fisher distance meaningfully larger than 0 for the pair? | |
| Is the Fisher distance more discriminative than Euclidean? | |
| Overall: keep as Tier 1? | |